# Step 1.2a — Enriched Feature Matrix
**Thesis: Geopolitical Turning Points and Macroeconomic Volatility — Extension of Saadaoui (2026)**

Builds the high-dimensional control matrix for the DoubleML extension (Step 3).

**Changes from previous version:**
- Fixed Brent loading bug (was reading old Yahoo Finance file, now reads FRED correctly)
- Fixed `controls_baseline` (now matches Saadaoui's exact specification)
- Added: CNY/USD exchange rate, CNY/USD volatility, Dollar Index (DXY), US credit spread
- Added optional manual CDS controls (`us_cds_5y`, `china_cds_5y`) if CSVs are provided in raw data
- Main macro variables are sourced from FRED (credible, free, documented)


---
## Setup

In [1]:
import pandas as pd
import numpy as np
import os
import json
import requests
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pyreadstat
from pathlib import Path

# Paths (robust to kernel cwd)
cwd = Path.cwd().resolve()
if (cwd / 'notebooks').exists() and (cwd / 'data').exists():
    project_root = cwd
elif cwd.name == 'notebooks' and (cwd.parent / 'data').exists():
    project_root = cwd.parent
else:
    project_root = cwd

RAW  = str((project_root / 'data' / '02_features' / 'raw').resolve())
PROC = str((project_root / 'data' / '02_features').resolve())
DTA  = str((project_root / 'data' / 'Saadaoui_2026_JCE.dta').resolve())

os.makedirs(RAW, exist_ok=True)
os.makedirs(PROC, exist_ok=True)

# Sample window (buffer before 1990-01 allows lags to be constructed)
START = '1989-01-01'
END   = '2022-03-01'

print('Setup complete.')
print(f'PROJECT_ROOT → {project_root}')
print(f'RAW         → {RAW}')
print(f'PROC        → {PROC}')


Setup complete.
PROJECT_ROOT → C:\Users\HP\Desktop\replication+contribution
RAW         → C:\Users\HP\Desktop\replication+contribution\data\02_features\raw
PROC        → C:\Users\HP\Desktop\replication+contribution\data\02_features


---
## Section A — Load & Download Raw Data
Each cell skips the download if its output file already exists. Safe to re-run.


### A1. Saadaoui Core Dataset
**Source:** Saadaoui (2026), *Journal of Comparative Economics*
**File:** `../data/Saadaoui_2026_JCE.dta`


In [2]:
if not os.path.exists(DTA):
    raise FileNotFoundError(f'Core dataset not found: {DTA}')

df_core, _ = pyreadstat.read_dta(DTA)

# Convert Stata monthly date (%tm = months since Jan 1960) to datetime
base = pd.Timestamp('1960-01-01')
df_core['date'] = df_core['Period'].apply(
    lambda m: base + pd.DateOffset(months=int(m))
)
df_core = df_core.set_index('date').sort_index()
df_core.index = df_core.index.to_period('M').to_timestamp()

print(f'Core dataset: {df_core.shape[0]} obs, {df_core.shape[1]} variables')
print(f'Range: {df_core.index[0].date()} to {df_core.index[-1].date()}')


Core dataset: 386 obs, 48 variables
Range: 1990-01-01 to 2022-02-01


### A2. VIX — CBOE Volatility Index
**Source:** Yahoo Finance `^VIX`
**Why:** Captures global financial stress and risk appetite, which transmit geopolitical shocks to commodity markets (Bloom 2009, Hamilton & Lin 1996).


In [3]:
P_VIX = f'{RAW}/vix.csv'

if not os.path.exists(P_VIX):
    vix = yf.download('^VIX', start=START, end=END, progress=False)
    if len(vix) > 50:
        if isinstance(vix.columns, pd.MultiIndex):
            vix.columns = [c[0] for c in vix.columns]
        vix[['Close']].rename(columns={'Close': 'vix'}).to_csv(P_VIX)
        print(f'Downloaded VIX: {len(vix)} daily rows')
    else:
        print('VIX download failed')
else:
    print('VIX: file exists, skipping.')


VIX: file exists, skipping.


### A3. Gold Price (USD/oz)
**Source:** Datahub.io — LBMA monthly fixings
**Why:** Safe-haven asset; gold prices rise during geopolitical uncertainty and serve as an alternative channel for geopolitical shock transmission (Baur & Lucey 2010).


In [4]:
P_GOLD = f'{RAW}/gold_monthly.csv'

if not os.path.exists(P_GOLD):
    url = 'https://datahub.io/core/gold-prices/_r/-/data/monthly-processed.csv'
    try:
        r = requests.get(url, timeout=20)
        if r.status_code == 200:
            with open(P_GOLD, 'w') as fh:
                fh.write(r.text)
            print('Downloaded Gold: monthly LBMA data')
        else:
            print(f'Gold download failed: HTTP {r.status_code}')
    except Exception as e:
        print(f'Gold download error: {e}')
else:
    print('Gold: file exists, skipping.')


Gold: file exists, skipping.


### A4. Brent Crude Price
**Source:** FRED, `DCOILBRENTEU` (Europe Brent Spot Price FOB, USD/barrel)
**URL:** https://fred.stlouisfed.org/series/DCOILBRENTEU
**Coverage:** 1987-05-20 to present (full sample coverage for this study)

**Why Brent and not WTI?**
WTI (`lwti`) is the outcome variable. Including raw Brent as a control would introduce near-perfect collinearity (correlation ~0.99). Instead we include the **log Brent-WTI spread** (`lbrent_wti_spread`), which captures market segmentation between US domestic crude and global crude benchmarks. Geopolitical shocks affecting global supply chains widen this spread independently of WTI levels (Kilian & Murphy 2014). This is economically meaningful and orthogonal to the outcome.

**Fix from previous version:** The old code downloaded from Yahoo Finance (BZ=F, starts 2007) and saved to `brent.csv`. The merge cell (B2) still looked for `brent.csv`, so the FRED version was never used. This version loads directly from FRED into `brent_fred.csv` and the merge cell reads `brent_fred.csv`.


In [5]:
P_BRENT = f'{RAW}/brent_fred.csv'

if not os.path.exists(P_BRENT):
    url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU'
    try:
        brent_df = pd.read_csv(url, index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
        brent_df = brent_df[~brent_df.index.duplicated(keep='first')]
        brent_df['DCOILBRENTEU'] = pd.to_numeric(brent_df['DCOILBRENTEU'], errors='coerce')
        brent_df = brent_df.dropna(subset=['DCOILBRENTEU'])
        brent_df[['DCOILBRENTEU']].to_csv(P_BRENT)
        print(f'Downloaded Brent from FRED: {len(brent_df)} daily rows')
        print(f'Coverage: {brent_df.index[0].date()} to {brent_df.index[-1].date()}')
    except Exception as e:
        print(f'FRED Brent download failed: {e}')
else:
    print('Brent (FRED): file exists, skipping.')


Brent (FRED): file exists, skipping.


### A5. FRED Series — Treasury Yields, TED Spread, REER, Industrial Production
**Source:** FRED direct CSV — no API key required
**Why each:**
- `tb3ms`: 3-month T-bill rate — US monetary policy stance
- `gs10`: 10-year Treasury yield — long-run real interest rate channel
- `tedrate`: TED spread (Libor–T-bill) — interbank credit risk, financial stress
- `reer_bis`: BIS Broad REER — global dollar conditions affecting oil pricing
- `indpro`: US industrial production — oil demand proxy (Kilian 2009)


In [6]:
FRED_SERIES = {
    'tb3ms'   : f'{RAW}/tb3ms.csv',
    'gs10'    : f'{RAW}/gs10.csv',
    'tedrate' : f'{RAW}/tedrate.csv',
    'reer_bis': f'{RAW}/reer_bis.csv',
    'indpro'  : f'{RAW}/indpro.csv',
}

FRED_IDS = {
    'tb3ms'   : 'TB3MS',
    'gs10'    : 'GS10',
    'tedrate' : 'TEDRATE',
    'reer_bis': 'RBUSBIS',
    'indpro'  : 'INDPRO',
}

for key, path in FRED_SERIES.items():
    if not os.path.exists(path):
        fred_id = FRED_IDS[key]
        url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={fred_id}'
        try:
            r = requests.get(url, timeout=20)
            if r.status_code == 200:
                with open(path, 'w') as fh:
                    fh.write(r.text)
                print(f'Downloaded {key} ({fred_id})')
            else:
                print(f'{key}: HTTP {r.status_code}')
        except Exception as e:
            print(f'{key}: {e}')
    else:
        print(f'{key}: file exists, skipping.')


tb3ms: file exists, skipping.
gs10: file exists, skipping.
tedrate: file exists, skipping.
reer_bis: file exists, skipping.
indpro: file exists, skipping.


### A6. CNY/USD, Dollar Index, Credit Spreads, and Financial Risk Controls
**Source:** FRED auto-downloads (plus optional manual CDS files)

**FX and dollar controls**
- `DEXCHUS` (CNY/USD)
- `DTWEXBGS` (broad dollar index)

**Credit and financial stress controls (auto from FRED)**
- `BAA10Y` (US credit spread; already Baa minus 10Y Treasury)
- `BAMLC0A0CM` (US investment-grade corporate OAS)
- `BAMLC0A4CBBB` (US BBB corporate OAS)
- `BAMLEMCBPIOAS` (emerging market corporate OAS)

**Optional manual files (if you have them)**
- `us_cds_5y.csv`
- `china_cds_5y.csv`


In [7]:
### CNY/USD, Dollar Index, Credit Spreads, and Financial Controls
# ── Auto-download required FRED files ──────────────────────────────────────────
FRED_FILES = {
    'cny_usd': ('DEXCHUS', f'{RAW}/cny_usd.csv'),
    'dxy': ('DTWEXBGS', f'{RAW}/dxy.csv'),
    'baa10y': ('BAA10Y', f'{RAW}/baa10y.csv'),
    'us_ig_oas': ('BAMLC0A0CM', f'{RAW}/us_ig_oas.csv'),
    'us_bbb_oas': ('BAMLC0A4CBBB', f'{RAW}/us_bbb_oas.csv'),
    'em_oas': ('BAMLEMCBPIOAS', f'{RAW}/em_oas.csv'),
}

for key, (fred_id, path) in FRED_FILES.items():
    if not os.path.exists(path):
        url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={fred_id}'
        try:
            r = requests.get(url, timeout=20)
            if r.status_code == 200:
                with open(path, 'w') as fh:
                    fh.write(r.text)
                print(f'Downloaded {key} ({fred_id})')
            else:
                print(f'{key}: HTTP {r.status_code}')
        except Exception as e:
            print(f'{key}: {e}')
    else:
        print(f'{key}: file exists, skipping.')

# Build canonical us_spread from BAA10Y (already Baa - 10Y spread)
P_BAA10Y = f'{RAW}/baa10y.csv'
if os.path.exists(P_BAA10Y):
    baa10y_df = pd.read_csv(P_BAA10Y, index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
    baa10y_df.index = pd.to_datetime(baa10y_df.index, errors='coerce')
    baa10y_df = baa10y_df[baa10y_df.index.notna()]
    baa10y_df = baa10y_df[~baa10y_df.index.duplicated(keep='first')]
    us_spread = pd.to_numeric(baa10y_df.iloc[:, 0], errors='coerce').resample('MS').last().rename('us_spread')
    pd.DataFrame({'us_spread': us_spread}).to_csv(f'{RAW}/us_spread.csv')
    print(f'Computed us_spread (BAA10Y direct): {us_spread.notna().sum()} months')

# ── Optional CDS controls (manual CSVs, if available) ─────────────────────────
# Keep these in RAW folder as: us_cds_5y.csv and china_cds_5y.csv
# Expected schema: first column = date, second column = CDS value (bps or percent).
CDS_FILES = {
    'us_cds_5y': f'{RAW}/us_cds_5y.csv',
    'china_cds_5y': f'{RAW}/china_cds_5y.csv',
}

for name, path in CDS_FILES.items():
    if os.path.exists(path):
        try:
            cds_df = pd.read_csv(path)
            cds_df = cds_df.iloc[:, :2].copy()
            cds_df.columns = ['date', name]
            cds_df['date'] = pd.to_datetime(cds_df['date'], errors='coerce', dayfirst=True)
            cds_df[name] = pd.to_numeric(
                cds_df[name].astype(str).str.replace(',', '.', regex=False),
                errors='coerce'
            )
            cds_df = cds_df.dropna(subset=['date']).set_index('date').sort_index()
            cds_m = cds_df[name].resample('MS').last().rename(name)
            cds_m.to_csv(path)
            print(f'{name}: {cds_m.notna().sum()} months | saved cleaned {path}')
        except Exception as e:
            print(f'{name}: parse failed ({e})')
    else:
        print(f'{name}: not found (optional)')

cny_usd: file exists, skipping.
dxy: file exists, skipping.
baa10y: file exists, skipping.
us_ig_oas: file exists, skipping.
us_bbb_oas: file exists, skipping.
em_oas: file exists, skipping.
Computed us_spread (BAA10Y direct): 484 months
us_cds_5y: not found (optional)
china_cds_5y: not found (optional)


### A7. Baltic Dry Index (BDI)
**Source:** Investing.com — manual download required
**Place file at:** `../data/02_features/raw/bdi.csv`
**URL:** https://www.investing.com/indices/baltic-dry-historical-data
**Why:** BDI captures global shipping demand, which reflects world trade activity and is a real-time indicator of commodity supply chain conditions (Kilian 2009; Alizadeh & Muradoglu 2014).


In [8]:
P_BDI     = f'{RAW}/bdi.csv'
P_BDI_OUT = f'{RAW}/bdi_clean.csv'

if os.path.exists(P_BDI):
    bdi_raw = pd.read_csv(P_BDI, sep=';', header=0, names=['date', 'bdi'],
                           decimal=',', thousands='.', na_values=['', ' ', '-'])
    if 'date' not in bdi_raw.columns or bdi_raw['date'].isna().all():
        bdi_raw = pd.read_csv(P_BDI, header=0)
        bdi_raw.columns = ['date', 'bdi'] + list(bdi_raw.columns[2:])
    bdi_raw['date'] = pd.to_datetime(bdi_raw['date'], dayfirst=True, errors='coerce')
    bdi_raw = bdi_raw.dropna(subset=['date']).set_index('date').sort_index()
    bdi_raw['bdi'] = pd.to_numeric(bdi_raw['bdi'].astype(str)
                                    .str.replace(',', '').str.strip(), errors='coerce')
    bdi_raw[['bdi']].to_csv(P_BDI_OUT)
    print(f'BDI: {bdi_raw["bdi"].notna().sum()} rows | '
          f'{bdi_raw.index[0].date()} to {bdi_raw.index[-1].date()}')
else:
    print('BDI file not found — download manually from Investing.com')
    print('Expected path:', os.path.abspath(P_BDI))


BDI: 9479 rows | 1986-12-31 to 2025-05-02


### A8. Global Supply Chain Pressure Index (GSCPI)
#
**Source:** Federal Reserve Bank of New York
**URL:** https://www.newyorkfed.org/research/policy/gscpi
**Why:** Measures global supply chain disruptions. Geopolitical turning points (trade wars, sanctions) directly cause supply chain stress, making this a key transmission variable (Benigno et al. 2022).
**Coverage:** 1998-01 to present (monthly)
**Format:** semicolon-delimited, French dates, comma decimals


In [9]:
# Paths
P_GSCPI_RAW = f'{RAW}/gscpi.csv'
P_GSCPI_CLEANED = f'{RAW}/gscpi_cleaned.csv'
gscpi_series = None

if os.path.exists(P_GSCPI_RAW):
    # 1. Read the raw file (keep all rows; previous skiprows dropped one month)
    gscpi_df = pd.read_csv(P_GSCPI_RAW, sep=';', header=0,
                           usecols=[0, 1],
                           na_values=['', ' ', '-'])

    date_col = gscpi_df.columns[0]
    value_col = gscpi_df.columns[1]
    
    # 2. Date parsing (French to English)
    FR_TO_EN = {
        'janv': 'Jan', 'fév': 'Feb', 'févr': 'Feb',
        'mars': 'Mar', 'avr': 'Apr', 'mai': 'May',
        'juin': 'Jun', 'juil': 'Jul', 'août': 'Aug', 'aout': 'Aug',
        'sept': 'Sep', 'oct': 'Oct', 'nov': 'Nov',
        'déc': 'Dec', 'dec': 'Dec',
    }
    
    def parse_gscpi_date(s):
        s = str(s).strip().lower()
        for fr, en in FR_TO_EN.items():
            s = s.replace(fr, en)
        s = s.title()
        dt = pd.to_datetime(s, dayfirst=True, errors='coerce')
        if pd.notna(dt) and dt.year > 2030:
            dt = dt.replace(year=dt.year - 100)
        return dt
    
    gscpi_df[date_col] = gscpi_df[date_col].apply(parse_gscpi_date)
    
    # 3. Handle comma decimals
    gscpi_df[value_col] = (gscpi_df[value_col].astype(str)
                            .str.replace(',', '.', regex=False)
                            .pipe(pd.to_numeric, errors='coerce'))
    
    # 4. Clean and resample
    gscpi_df = gscpi_df.dropna(subset=[date_col, value_col]).set_index(date_col).sort_index()
    gscpi_series = gscpi_df[value_col].resample('MS').last().rename('gscpi')
    
    # 5. SAVE TO CLEANED FILE (Leaving original untouched)
    gscpi_series.to_csv(P_GSCPI_CLEANED)
    
    # Update P_GSCPI so downstream cells use the cleaned file
    P_GSCPI = P_GSCPI_CLEANED
    
    n_valid = gscpi_series.notna().sum()
    print(f'GSCPI: {n_valid} months | {gscpi_series.first_valid_index().date()} → {gscpi_series.last_valid_index().date()}')
    print(f'✅ Processed. Original kept at {P_GSCPI_RAW}. Cleaned version saved to {P_GSCPI_CLEANED}.')

else:
    print(f'Original GSCPI file not found at {P_GSCPI_RAW}')

GSCPI: 339 months | 1998-01-01 → 2026-03-01
✅ Processed. Original kept at C:\Users\HP\Desktop\replication+contribution\data\02_features\raw/gscpi.csv. Cleaned version saved to C:\Users\HP\Desktop\replication+contribution\data\02_features\raw/gscpi_cleaned.csv.


### A9. Global Economic Policy Uncertainty Index (EPU)
**Source:** Baker, Bloom & Davis — policyuncertainty.com
**URL:** https://www.policyuncertainty.com/media/Global_Policy_Uncertainty_Data.xlsx
**Why:** Policy uncertainty is directly related to geopolitical events and affects investment and commodity demand. Baker, Bloom & Davis (2016) show EPU has strong predictive content for economic volatility.


In [10]:
P_EPU = f'{RAW}/epu_global.xlsx'

if not os.path.exists(P_EPU):
    url = 'https://www.policyuncertainty.com/media/Global_Policy_Uncertainty_Data.xlsx'
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            with open(P_EPU, 'wb') as fh:
                fh.write(r.content)
            print('Downloaded EPU global index')
        else:
            print(f'EPU download failed: HTTP {r.status_code}')
    except Exception as e:
        print(f'EPU download error: {e}')
else:
    print('EPU: file exists, skipping.')


EPU: file exists, skipping.


---
## Section B — Merge & Build Feature Matrix

### B1. Helper Functions

In [11]:
def ensure_datetimeindex(s):
    s = s.copy()
    if not isinstance(s.index, pd.DatetimeIndex):
        s.index = pd.to_datetime(s.index, errors='coerce')
    s = s[s.index.notna()]
    s = s[~s.index.duplicated(keep='first')]
    return s.sort_index()

def to_monthly_mean(s):
    s = ensure_datetimeindex(s)
    s = pd.to_numeric(s, errors='coerce')
    return s.resample('MS').mean()

def to_monthly_last(s):
    s = ensure_datetimeindex(s)
    s = pd.to_numeric(s, errors='coerce')
    return s.resample('MS').last()

def load_fred(path, col_name):
    df = pd.read_csv(path, index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
    df = ensure_datetimeindex(df)
    s  = pd.to_numeric(df.iloc[:, 0], errors='coerce')
    s  = s.resample('MS').last()
    return s.rename(col_name)

print('Helper functions defined.')


Helper functions defined.


### B2. Build All Monthly Series

In [ ]:
monthly_series = {}

# ── VIX ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/vix.csv'):
    vix_d = pd.read_csv(f'{RAW}/vix.csv', index_col=0,
                         skiprows=lambda i: i in [1, 2], na_values=['.','NA',''])
    vix_d = ensure_datetimeindex(vix_d)
    pcol  = [c for c in vix_d.columns if c.lower() in ('close','vix','adj close')][0]
    monthly_series['vix'] = to_monthly_mean(vix_d[pcol]).rename('vix')
    print(f"vix        : {monthly_series['vix'].notna().sum()} months")

# ── Gold ──────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/gold_monthly.csv'):
    gold_d = pd.read_csv(f'{RAW}/gold_monthly.csv', index_col=0, parse_dates=True)
    gold_d = ensure_datetimeindex(gold_d)
    gold_s = to_monthly_last(gold_d.iloc[:, 0])
    monthly_series['lgold'] = np.log(gold_s.replace(0, np.nan)).rename('lgold')
    print(f"lgold      : {monthly_series['lgold'].notna().sum()} months")

# ── Brent (FRED) — reads brent_fred.csv, NOT the old brent.csv ───────────────
if os.path.exists(f'{RAW}/brent_fred.csv'):
    brent_d = pd.read_csv(f'{RAW}/brent_fred.csv', index_col=0,
                           parse_dates=True, na_values=['.','NA',''])
    brent_d = ensure_datetimeindex(brent_d)
    brent_d['DCOILBRENTEU'] = pd.to_numeric(brent_d['DCOILBRENTEU'], errors='coerce')
    brent_d = brent_d.dropna(subset=['DCOILBRENTEU'])
    brent_m = to_monthly_mean(brent_d['DCOILBRENTEU'])
    monthly_series['lbrent'] = np.log(brent_m.replace(0, np.nan)).rename('lbrent')
    print(f"lbrent     : {monthly_series['lbrent'].notna().sum()} months (full raw history)")
else:
    print("lbrent     : brent_fred.csv not found — run A4 first")

# ── FRED core series ──────────────────────────────────────────────────────────
fred_load = {
    'tb3ms'   : f'{RAW}/tb3ms.csv',
    'gs10'    : f'{RAW}/gs10.csv',
    'tedrate' : f'{RAW}/tedrate.csv',
    'reer'    : f'{RAW}/reer_bis.csv',
    'indpro'  : f'{RAW}/indpro.csv',
}
for col, path in fred_load.items():
    if os.path.exists(path):
        monthly_series[col] = load_fred(path, col)
        print(f"{col:10s}: {monthly_series[col].notna().sum()} months")

# ── CNY/USD exchange rate ────────────────────────────────────────────────
if os.path.exists(f'{RAW}/cny_usd.csv'):
    cny = load_fred(f'{RAW}/cny_usd.csv', 'cny_usd')
    monthly_series['cny_usd'] = cny
    print(f"cny_usd    : {cny.notna().sum()} months")

# ── Dollar Index (DXY equivalent) ───────────────────────────────────────
if os.path.exists(f'{RAW}/dxy.csv'):
    dxy = load_fred(f'{RAW}/dxy.csv', 'dxy')
    monthly_series['dxy'] = dxy
    print(f"dxy        : {dxy.notna().sum()} months")

# ── US Credit Spread (BAA10Y is already Baa - 10Y spread) ─────────────────────
if os.path.exists(f'{RAW}/baa10y.csv'):
    cs = load_fred(f'{RAW}/baa10y.csv', 'us_spread')
    monthly_series['us_spread'] = cs
    print(f"us_spread  : {cs.notna().sum()} months (BAA10Y direct)")
else:
    print("us_spread  : missing baa10y.csv")

# ── Additional financial controls from FRED ────────────────────────────────────
for fin_col, fin_file in {
    'us_ig_oas': f'{RAW}/us_ig_oas.csv',
    'us_bbb_oas': f'{RAW}/us_bbb_oas.csv',
    'em_oas': f'{RAW}/em_oas.csv',
}.items():
    if os.path.exists(fin_file):
        monthly_series[fin_col] = load_fred(fin_file, fin_col)
        print(f"{fin_col:10s}: {monthly_series[fin_col].notna().sum()} months")

# Optional manual CDS files (if you have them locally)
for cds_name in ['us_cds_5y', 'china_cds_5y']:
    p_cds = f'{RAW}/{cds_name}.csv'
    if os.path.exists(p_cds):
        try:
            cds_df = pd.read_csv(p_cds, index_col=0, parse_dates=True, na_values=['.', 'NA', ''])
            cds_df = ensure_datetimeindex(cds_df)
            cds_s = pd.to_numeric(cds_df.iloc[:, 0], errors='coerce').resample('MS').last().rename(cds_name)
            monthly_series[cds_name] = cds_s
            print(f"{cds_name:10s}: {cds_s.notna().sum()} months")
        except Exception as e:
            print(f"{cds_name:10s}: load error ({e})")

# ── BDI ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/bdi_clean.csv'):
    bdi_d = pd.read_csv(f'{RAW}/bdi_clean.csv', index_col=0, parse_dates=True)
    bdi_d = ensure_datetimeindex(bdi_d)
    monthly_series['bdi'] = to_monthly_mean(bdi_d['bdi']).rename('bdi')
    print(f"bdi        : {monthly_series['bdi'].notna().sum()} months")

# ── GSCPI ─────────────────────────────────────────────────────────────────────
if gscpi_series is not None and gscpi_series.notna().any():
    monthly_series['gscpi'] = gscpi_series
    print(f"gscpi      : {monthly_series['gscpi'].notna().sum()} months")

# ── EPU ───────────────────────────────────────────────────────────────────────
if os.path.exists(f'{RAW}/epu_global.xlsx'):
    try:
        epu_df = pd.read_excel(f'{RAW}/epu_global.xlsx', header=None)
        header_row = epu_df[epu_df[0] == 'Year'].index[0]
        epu_df = pd.read_excel(f'{RAW}/epu_global.xlsx', skiprows=header_row)
        epu_df.columns = ['Year', 'Month', 'GEPU_current', 'GEPU_ppp']
        epu_df['Year']  = pd.to_numeric(epu_df['Year'],  errors='coerce')
        epu_df['Month'] = pd.to_numeric(epu_df['Month'], errors='coerce')
        epu_df = epu_df.dropna(subset=['Year', 'Month'])
        epu_df['date'] = pd.to_datetime(epu_df[['Year','Month']].assign(Day=1))
        epu_col = 'GEPU_ppp' if 'GEPU_ppp' in epu_df.columns else 'GEPU_current'
        epu_s = epu_df.set_index('date')[epu_col].sort_index()
        epu_s = ensure_datetimeindex(epu_s)
        monthly_series['epu'] = epu_s.resample('MS').last().rename('epu')
        print(f"epu        : {monthly_series['epu'].notna().sum()} months")
    except Exception as e:
        print(f"EPU load error: {e}")

print(f'\nTotal series collected: {len(monthly_series)}')


vix        : 386 months
lgold      : 2319 months
lbrent     : 468 months (full raw history)
tb3ms     : 1107 months
gs10      : 876 months
tedrate   : 433 months
reer      : 386 months
indpro    : 1287 months
cny_usd    : 544 months
dxy        : 244 months
us_spread  : 484 months (BAA10Y direct)
us_ig_oas : 37 months
us_bbb_oas: 37 months
em_oas    : 37 months
bdi        : 462 months
gscpi      : 339 months
epu        : 347 months

Total series collected: 17


### B3. Derived Variables

In [13]:
derived = []

if 'gs10' in monthly_series and 'tb3ms' in monthly_series:
    monthly_series['term_spread'] = (monthly_series['gs10'] - monthly_series['tb3ms']).rename('term_spread')
    derived.append('term_spread')

if 'reer' in monthly_series:
    monthly_series['lreer'] = np.log(monthly_series['reer'].replace(0, np.nan)).rename('lreer')
    derived.append('lreer')

if 'vix' in monthly_series:
    monthly_series['dvix'] = monthly_series['vix'].diff().rename('dvix')
    derived.append('dvix')

if 'bdi' in monthly_series:
    monthly_series['lbdi'] = np.log(monthly_series['bdi'].replace(0, np.nan)).rename('lbdi')
    derived.append('lbdi')

if 'indpro' in monthly_series:
    monthly_series['lip'] = np.log(monthly_series['indpro'].replace(0, np.nan)).rename('lip')
    derived.append('lip')

if 'epu' in monthly_series:
    monthly_series['lepu'] = np.log(monthly_series['epu'].replace(0, np.nan)).rename('lepu')
    derived.append('lepu')

# CNY/USD volatility — rolling 3-month std of monthly log changes
# Captures FX risk transmission from US-China geopolitical shocks
if 'cny_usd' in monthly_series:
    cny_ret = monthly_series['cny_usd'].apply(np.log).diff()
    monthly_series['cny_vol'] = cny_ret.rolling(3, min_periods=2).std().rename('cny_vol')
    derived.append('cny_vol')

# Log Dollar Index
if 'dxy' in monthly_series:
    monthly_series['ldxy'] = np.log(monthly_series['dxy'].replace(0, np.nan)).rename('ldxy')
    derived.append('ldxy')

# Brent-WTI spread (log difference)
# Used instead of raw lbrent to avoid near-perfect collinearity with the outcome (lwti)
# The spread captures global vs US market segmentation and geopolitical supply disruptions
if 'lbrent' in monthly_series:
    # We merge with core to get lwti for the spread
    lwti_s = ensure_datetimeindex(df_core['lwti'].copy())
    lwti_s.index = lwti_s.index.to_period('M').to_timestamp()
    lbrent_aligned = monthly_series['lbrent'].reindex(lwti_s.index)
    monthly_series['brent_wti_spread'] = (lbrent_aligned - lwti_s).rename('brent_wti_spread')
    derived.append('brent_wti_spread')

print(f'Derived variables: {derived}')


Derived variables: ['term_spread', 'lreer', 'dvix', 'lbdi', 'lip', 'lepu', 'cny_vol', 'ldxy', 'brent_wti_spread']


### B4. Merge with Core Dataset
> **Look-ahead bias prevention:** All new macro controls are lagged 1 month (`shift(1)`) before merging.
> At time t, only information available up to t-1 is used as controls.
> This is critical for causal validity of the DoubleML step.


In [14]:
df_core.index = pd.to_datetime(df_core.index).to_period('M').to_timestamp()
df_enriched = df_core.copy()

for col_name, series in monthly_series.items():
    s = series.copy()
    s = ensure_datetimeindex(s)
    s.index = s.index.to_period('M').to_timestamp()
    s.name = col_name
    df_enriched = df_enriched.join(s, how='left')

# Lag all new macro controls by 1 month (look-ahead bias prevention)
new_ctrl_cols = [c for c in monthly_series.keys() if c in df_enriched.columns]
df_enriched[new_ctrl_cols] = df_enriched[new_ctrl_cols].shift(1)

# Restrict to Saadaoui sample
df_enriched = df_enriched.loc['1990-01-01':'2022-01-01']

print(f'Enriched dataset: {df_enriched.shape[0]} obs × {df_enriched.shape[1]} variables')
print(f'Range: {df_enriched.index[0].date()} to {df_enriched.index[-1].date()}')


Enriched dataset: 385 obs × 74 variables
Range: 1990-01-01 to 2022-01-01


### B5. Coverage Audit

In [15]:
new_vars = [c for c in monthly_series.keys() if c in df_enriched.columns]

audit = pd.DataFrame({
    'n_obs'      : df_enriched[new_vars].notna().sum(),
    'pct_filled' : (df_enriched[new_vars].notna().mean() * 100).round(1),
    'first_obs'  : df_enriched[new_vars].apply(lambda c: c.first_valid_index()),
    'last_obs'   : df_enriched[new_vars].apply(lambda c: c.last_valid_index()),
}).sort_values('pct_filled', ascending=False)

print(audit.to_string())

sparse = audit[audit['pct_filled'] < 80]
if len(sparse) > 0:
    print(f'\nVariables <80% coverage:')
    print(sparse[['pct_filled', 'first_obs']].to_string())
else:
    print('\nAll variables have ≥80% coverage.')


                  n_obs  pct_filled  first_obs   last_obs
vix                 384        99.7 1990-02-01 2022-01-01
lgold               384        99.7 1990-02-01 2022-01-01
lbrent              384        99.7 1990-02-01 2022-01-01
tb3ms               384        99.7 1990-02-01 2022-01-01
gs10                384        99.7 1990-02-01 2022-01-01
tedrate             384        99.7 1990-02-01 2022-01-01
indpro              384        99.7 1990-02-01 2022-01-01
cny_usd             384        99.7 1990-02-01 2022-01-01
bdi                 384        99.7 1990-02-01 2022-01-01
us_spread           384        99.7 1990-02-01 2022-01-01
cny_vol             384        99.7 1990-02-01 2022-01-01
brent_wti_spread    384        99.7 1990-02-01 2022-01-01
term_spread         384        99.7 1990-02-01 2022-01-01
lip                 384        99.7 1990-02-01 2022-01-01
lbdi                384        99.7 1990-02-01 2022-01-01
dvix                383        99.5 1990-03-01 2022-01-01
lreer         

### B6. Variable Role Dictionary

In [16]:
VAR_ROLES = {
    'outcome'            : ['lwti'],
    'treatment'          : ['lpri'],
    'instrument'         : ['d2pri', 'd2pri_jp'],
    # llwip  = lag 1 of log world industrial production
    # dllgop = first difference of lag 1 of log global oil production
    # l2lwip = lag 2 of log world industrial production
    # dl2lgop = first difference of lag 2 of log global oil production
    # These match Stata: llwip d.llgop l2lwip d.l2lgop
    'controls_baseline'  : ['llwip', 'dllgop', 'l2lwip', 'dl2lgop'],
    'controls_alliance'  : [c for c in df_enriched.columns if c.startswith('dlpri_')
                             or (c.startswith('d_lpri_') and c in df_enriched.columns)],
    'controls_macro'     : [
        c for c in [
            'vix', 'dvix', 'gs10', 'tb3ms', 'term_spread', 'tedrate',
            'lreer', 'lgold', 'lbdi', 'lip', 'gscpi', 'lepu',
            'brent_wti_spread',    # Brent-WTI spread (not raw lbrent)
            'cny_usd', 'cny_vol',  # CNY/USD level and volatility
            'ldxy',                # log Dollar Index
            'us_spread',           # US credit spread (BAA10Y)
            'us_ig_oas',           # US investment-grade OAS
            'us_bbb_oas',          # US BBB OAS
            'em_oas',              # emerging-market OAS
            'us_cds_5y',           # optional US 5Y CDS (if file provided)
            'china_cds_5y',        # optional China 5Y CDS (if file provided)
        ]
        if c in df_enriched.columns
    ],
}

VAR_ROLES['controls_all_ml'] = (
    VAR_ROLES['controls_baseline']
    + VAR_ROLES['controls_alliance']
    + VAR_ROLES['controls_macro']
)

print('Variable inventory:')
for role, cols in VAR_ROLES.items():
    print(f'  {role:25s} ({len(cols):2d}): {cols}')
print(f'\nTotal ML control variables: {len(VAR_ROLES["controls_all_ml"])}')


Variable inventory:
  outcome                   ( 1): ['lwti']
  treatment                 ( 1): ['lpri']
  instrument                ( 2): ['d2pri', 'd2pri_jp']
  controls_baseline         ( 4): ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']
  controls_alliance         (11): ['dlpri_jp', 'dlpri_aus', 'dlpri_cds', 'dlpri_fra', 'dlpri_ger', 'dlpri_india', 'dlpri_indo', 'dlpri_pak', 'dlpri_rus', 'dlpri_vn', 'dlpri_uk']
  controls_macro            (20): ['vix', 'dvix', 'gs10', 'tb3ms', 'term_spread', 'tedrate', 'lreer', 'lgold', 'lbdi', 'lip', 'gscpi', 'lepu', 'brent_wti_spread', 'cny_usd', 'cny_vol', 'ldxy', 'us_spread', 'us_ig_oas', 'us_bbb_oas', 'em_oas']
  controls_all_ml           (35): ['llwip', 'dllgop', 'l2lwip', 'dl2lgop', 'dlpri_jp', 'dlpri_aus', 'dlpri_cds', 'dlpri_fra', 'dlpri_ger', 'dlpri_india', 'dlpri_indo', 'dlpri_pak', 'dlpri_rus', 'dlpri_vn', 'dlpri_uk', 'vix', 'dvix', 'gs10', 'tb3ms', 'term_spread', 'tedrate', 'lreer', 'lgold', 'lbdi', 'lip', 'gscpi', 'lepu', 'brent_wti_sprea

### B7. Export

In [17]:
OUT_CSV   = f'{PROC}/feature_matrix.csv'
OUT_ROLES = f'{PROC}/var_roles.json'

df_enriched.to_csv(OUT_CSV)

# Convert lists in VAR_ROLES for JSON serialization
with open(OUT_ROLES, 'w') as fh:
    json.dump({k: list(v) for k, v in VAR_ROLES.items()}, fh, indent=2)

print(f'Saved: {OUT_CSV}')
print(f'Saved: {OUT_ROLES}')
print(f'Feature matrix shape: {df_enriched.shape}')


Saved: C:\Users\HP\Desktop\replication+contribution\data\02_features/feature_matrix.csv
Saved: C:\Users\HP\Desktop\replication+contribution\data\02_features/var_roles.json
Feature matrix shape: (385, 74)


---
## Descriptive Statistics & Correlations

In [18]:
macro_vars = VAR_ROLES.get('controls_macro', [])
if macro_vars:
    print('=== Descriptive statistics ===')
    display(df_enriched[macro_vars].describe().round(3))


=== Descriptive statistics ===


,vix,dvix,gs10,tb3ms,term_spread,tedrate,lreer,lgold,lbdi,lip,gscpi,lepu,brent_wti_spread,cny_usd,cny_vol,ldxy,us_spread,us_ig_oas,us_bbb_oas,em_oas
count,384.000,383.000,384.000,384.000,384.000,384.000,336.000,384.000,384.000,384.000,288.000,300.000,384.000,384.000,384.000,192.000,384.000,0.0,0.0,0.0
mean,19.490,-0.007,4.301,2.548,1.753,0.467,4.526,6.470,9.580,4.477,-0.052,4.747,0.654,7.153,0.007,4.621,2.350,NaN,NaN,NaN
std,7.665,4.158,2.036,2.232,1.090,0.356,0.082,0.672,0.575,0.161,0.944,0.456,0.278,1.068,0.023,0.104,0.729,NaN,NaN,NaN
min,10.125,-16.283,0.620,0.010,-0.530,0.060,4.362,5.545,7.938,4.100,-1.530,3.948,0.137,4.734,0.000,4.450,1.300,NaN,NaN,NaN
25%,13.973,-1.731,2.512,0.160,0.878,0.230,4.454,5.876,9.250,4.434,-0.620,4.364,0.383,6.355,0.000,4.532,1.790,NaN,NaN,NaN
50%,17.620,-0.276,4.210,2.085,1.695,0.380,4.540,6.272,9.662,4.530,-0.280,4.732,0.654,6.883,0.002,4.590,2.215,NaN,NaN,NaN
75%,23.099,1.182,5.905,4.812,2.662,0.572,4.586,7.145,10.048,4.597,0.143,5.047,0.920,8.277,0.006,4.725,2.762,NaN,NaN,NaN
max,62.639,38.108,8.890,7.900,3.760,3.150,4.689,7.585,10.369,4.645,4.490,6.036,1.099,8.730,0.234,4.808,6.100,NaN,NaN,NaN


In [19]:
if macro_vars:
    print('=== Correlations with log WTI ===')
    corr = (
        df_enriched[macro_vars + ['lwti']]
        .corr()['lwti']
        .drop('lwti')
        .sort_values(key=abs, ascending=False)
        .round(3)
    )
    print(corr.to_string())


=== Correlations with log WTI ===
ldxy               -0.878
lgold               0.652
brent_wti_spread    0.639
lip                 0.557
lreer              -0.555
lbdi                0.550
tb3ms              -0.499
gs10               -0.498
cny_usd            -0.324
us_spread           0.250
vix                -0.122
cny_vol            -0.100
term_spread         0.091
lepu                0.082
tedrate            -0.075
gscpi               0.070
dvix               -0.033
us_ig_oas             NaN
us_bbb_oas            NaN
em_oas                NaN
